In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
precision_score,
recall_score,
f1_score
)

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(

"/kaggle/input/datasets/asdasdasasdas/garbage-classification/Garbage classification/Garbage classification",

validation_split=0.2,

subset="training",

seed=42,

image_size=(224,224),

batch_size=32

)

val_ds = tf.keras.utils.image_dataset_from_directory(

"/kaggle/input/datasets/asdasdasasdas/garbage-classification/Garbage classification/Garbage classification",

validation_split=0.2,

subset="validation",

seed=42,

image_size=(224,224),

batch_size=32

)

num_classes = len(train_ds.class_names)

Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.


2026-06-24 07:25:09.790619: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
inputs = tf.keras.Input(
shape=(224,224,3)
)

x = tf.keras.layers.Rescaling(
1./255
)(inputs)

x = tf.keras.layers.Conv2D(
96,
4,
strides=4,
padding="same"
)(x)

x = tf.keras.layers.LayerNormalization()(x)

x = tf.keras.layers.Conv2D(
192,
2,
strides=2,
padding="same"
)(x)

x = tf.keras.layers.MultiHeadAttention(
num_heads=8,
key_dim=64
)(
x,
x
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dense(
256,
activation='relu'
)(x)

x = tf.keras.layers.Dropout(
0.3
)(x)

outputs = tf.keras.layers.Dense(
num_classes,
activation='softmax'
)(x)

model = tf.keras.Model(
inputs,
outputs
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 56, 56,    │      4,704 │ rescaling[0][0]   │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 56, 56,    │        192 │ conv2d[0][0]      │
│ (LayerNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 28, 28,    │     73,920 │ layer_normalizat… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 28, 28,    │    394,944 │ conv2d_1[0][0],   │
│ (MultiHeadAttentio… │ 192)              │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 192)       │          0 │ multi_head_atten… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     49,408 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 6)         │      1,542 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 524,710 (2.00 MB)

 Trainable params: 524,710 (2.00 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(

optimizer=tf.keras.optimizers.Adam(
learning_rate=0.0001
),

loss='sparse_categorical_crossentropy',

metrics=['accuracy']

)

In [7]:
history = model.fit(

train_ds,

validation_data=val_ds,

epochs=20

)

Epoch 1/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 392s 6s/step - accuracy: 0.2493 - loss: 1.7051 - val_accuracy: 0.2554 - val_loss: 1.6610
Epoch 2/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 353s 6s/step - accuracy: 0.3694 - loss: 1.5883 - val_accuracy: 0.3347 - val_loss: 1.5642
Epoch 3/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 355s 6s/step - accuracy: 0.3818 - loss: 1.5233 - val_accuracy: 0.3188 - val_loss: 1.5597
Epoch 4/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 354s 6s/step - accuracy: 0.4060 - loss: 1.4746 - val_accuracy: 0.3960 - val_loss: 1.4702
Epoch 5/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 361s 6s/step - accuracy: 0.4327 - loss: 1.4236 - val_accuracy: 0.4376 - val_loss: 1.4154
Epoch 6/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 382s 6s/step - accuracy: 0.4555 - loss: 1.3733 - val_accuracy: 0.4574 - val_loss: 1.3564
Epoch 7/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 360s 6s/step - accuracy: 0.4758 - loss: 1.3349 - val_accuracy: 0.4891 - val_loss: 1.3018
Epoch 8/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 358s 6s/step - accuracy: 0.4926 - loss: 1.2955 - val_accuracy: 0.4851 - v

In [8]:
loss, accuracy = model.evaluate(
val_ds
)

print(
"Accuracy:",
accuracy
)

16/16 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.5485 - loss: 1.0883
Accuracy: 0.5485148429870605
